[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IshtVibhu/iv-Mesa-ABM-Tutorial/blob/main/NB_10_Financial_Data.ipynb)

# Python for Computational Economics
## Notebook 10 — Financial Data: Prices, Returns, and Portfolio Analysis

**Prerequisites:** NB_01–NB_04

**What you will learn:**
- Downloading financial market data with `yfinance`
- Computing log-returns, volatility, and rolling correlations
- Modern Portfolio Theory: efficient frontier
- Value at Risk (VaR) and Expected Shortfall
- FRED macroeconomic data via `pandas-datareader`

**Note:** `yfinance` requires an internet connection. All cells fall back to simulated data if the download fails.

In [ ]:
try:
    import yfinance as yf
except ImportError:
    !pip install yfinance --quiet
    import yfinance as yf

try:
    import pandas_datareader as pdr
except ImportError:
    !pip install pandas-datareader --quiet
    import pandas_datareader as pdr

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded.")

---
## 1. Downloading Stock Data

`yfinance` downloads OHLCV (Open, High, Low, Close, Volume) data from Yahoo Finance.

In [ ]:
TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "JPM"]
START   = "2018-01-01"
END     = "2024-01-01"

try:
    prices = yf.download(TICKERS, start=START, end=END, auto_adjust=True)["Close"]
    prices = prices.dropna()
    print(f"Downloaded {len(prices)} trading days of data.")
except Exception as e:
    print(f"Download failed ({e}). Using simulated data.")
    # Simulate correlated log-normal price series
    np.random.seed(42)
    n = 1500
    dates = pd.date_range(START, periods=n, freq="B")
    corr = np.array([[1.0, 0.7, 0.6, 0.5, 0.3],
                     [0.7, 1.0, 0.7, 0.5, 0.3],
                     [0.6, 0.7, 1.0, 0.5, 0.2],
                     [0.5, 0.5, 0.5, 1.0, 0.2],
                     [0.3, 0.3, 0.2, 0.2, 1.0]])
    L = np.linalg.cholesky(corr)
    daily_returns = (L @ np.random.normal(0.0005, 0.015, (5, n))).T
    prices = pd.DataFrame(
        np.exp(np.cumsum(daily_returns, axis=0)) * np.array([150, 100, 120, 90, 110]),
        index=dates, columns=TICKERS
    )

print(prices.tail(3).round(2))

---
## 2. Log Returns, Volatility, and Correlations

In [ ]:
# Log returns: r_t = ln(P_t / P_{t-1})
log_returns = np.log(prices / prices.shift(1)).dropna()

# Annualised statistics (252 trading days per year)
annual_return  = log_returns.mean() * 252
annual_vol     = log_returns.std()  * np.sqrt(252)
sharpe_ratio   = annual_return / annual_vol

stats_df = pd.DataFrame({
    "Annual Return": annual_return,
    "Annual Vol":    annual_vol,
    "Sharpe Ratio":  sharpe_ratio,
}).round(3)

print("Annualised Return, Volatility, and Sharpe Ratio:")
print(stats_df)

# Correlation matrix
print("\nReturn Correlation Matrix:")
print(log_returns.corr().round(3))

In [ ]:
# Rolling 30-day volatility
rolling_vol = log_returns.rolling(30).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(12, 4))
for ticker in TICKERS:
    ax.plot(rolling_vol.index, rolling_vol[ticker] * 100, label=ticker, linewidth=1)
ax.set_xlabel("Date")
ax.set_ylabel("Annualised Volatility (%)")
ax.set_title("30-Day Rolling Volatility")
ax.legend()
plt.tight_layout()
plt.savefig("rolling_volatility.png", dpi=100, bbox_inches="tight")
plt.show()

---
## 3. Modern Portfolio Theory — Efficient Frontier

Markowitz (1952): for a given expected return, find the portfolio with **minimum variance**. The set of such portfolios traces out the **efficient frontier**.

In [ ]:
from scipy.optimize import minimize

mu  = log_returns.mean().values * 252          # annualised expected returns
Sigma = log_returns.cov().values * 252          # annualised covariance matrix
n_assets = len(TICKERS)

def portfolio_stats(weights):
    port_return = weights @ mu
    port_vol    = np.sqrt(weights @ Sigma @ weights)
    return port_return, port_vol

def min_vol_for_target(target_return):
    constraints = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1},
        {"type": "eq", "fun": lambda w: w @ mu - target_return},
    ]
    bounds = [(0, 1)] * n_assets
    result = minimize(
        lambda w: portfolio_stats(w)[1],
        x0=np.ones(n_assets) / n_assets,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
    )
    return result

# Trace efficient frontier
target_returns = np.linspace(mu.min(), mu.max(), 40)
frontier_vols  = []
frontier_rets  = []
for tr in target_returns:
    res = min_vol_for_target(tr)
    if res.success:
        r, v = portfolio_stats(res.x)
        frontier_rets.append(r * 100)
        frontier_vols.append(v * 100)

# Maximum Sharpe ratio portfolio
def neg_sharpe(w):
    r, v = portfolio_stats(w)
    return -(r - 0.03) / v  # risk-free rate 3%

res_sharpe = minimize(
    neg_sharpe,
    x0=np.ones(n_assets) / n_assets,
    method="SLSQP",
    bounds=[(0, 1)] * n_assets,
    constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1}],
)
msr_r, msr_v = portfolio_stats(res_sharpe.x)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(frontier_vols, frontier_rets, "b-", linewidth=2, label="Efficient frontier")
ax.scatter([v * 100 for v in [annual_vol[t] for t in TICKERS]],
           [r * 100 for r in [annual_return[t] for t in TICKERS]],
           s=80, zorder=5)
for i, ticker in enumerate(TICKERS):
    ax.annotate(ticker, (annual_vol[ticker]*100 + 0.3, annual_return[ticker]*100))
ax.scatter([msr_v * 100], [msr_r * 100], s=150, color="red", zorder=6,
           label=f"Max Sharpe (Sharpe={-res_sharpe.fun:.2f})")
ax.set_xlabel("Annualised Volatility (%)")
ax.set_ylabel("Annualised Return (%)")
ax.set_title("Efficient Frontier — Markowitz (1952)")
ax.legend()
plt.tight_layout()
plt.savefig("efficient_frontier.png", dpi=100, bbox_inches="tight")
plt.show()

print("Maximum Sharpe Ratio Portfolio Weights:")
for ticker, weight in zip(TICKERS, res_sharpe.x):
    print(f"  {ticker}: {weight:.1%}")

---
## 4. Value at Risk (VaR) and Expected Shortfall

**VaR at confidence level α:** the loss that will not be exceeded with probability α over a given horizon.

**Expected Shortfall (CVaR):** the *average* loss conditional on exceeding VaR. A more robust risk measure.

In [ ]:
# Equal-weight portfolio
weights_eq = np.ones(n_assets) / n_assets
port_returns = log_returns @ weights_eq

# Historical VaR and CVaR at 99% and 95%
for alpha in [0.99, 0.95]:
    var    = np.percentile(port_returns, (1 - alpha) * 100)
    es     = port_returns[port_returns <= var].mean()
    print(f"At {alpha:.0%} confidence level:")
    print(f"  1-day VaR:              {-var*100:.3f}% (lose at most this with prob {alpha:.0%})")
    print(f"  Expected Shortfall:     {-es*100:.3f}% (avg loss when VaR exceeded)")

# Plot return distribution with VaR
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(port_returns * 100, bins=80, color="steelblue", edgecolor="white", alpha=0.8, density=True)
var_99 = np.percentile(port_returns, 1) * 100
ax.axvline(var_99, color="red", linestyle="--", linewidth=2, label=f"99% VaR = {-var_99:.2f}%")
ax.set_xlabel("Daily Return (%)")
ax.set_ylabel("Density")
ax.set_title("Equal-Weight Portfolio Return Distribution")
ax.legend()
plt.tight_layout()
plt.savefig("var_distribution.png", dpi=100, bbox_inches="tight")
plt.show()

---
## Your Turn — Compare EW vs Maximum-Sharpe Portfolio Risk

Using the maximum-Sharpe weights (`res_sharpe.x`) computed above:
1. Compute the annualised return and volatility for both the equal-weight and max-Sharpe portfolios
2. Compute the 95% VaR for both portfolios
3. Print a comparison table

Does the higher Sharpe portfolio always have lower VaR?

In [ ]:
# Your code here


In [ ]:
#@title Solution
def portfolio_summary(weights, name):
    pr = log_returns @ weights
    ann_r = pr.mean() * 252 * 100
    ann_v = pr.std() * np.sqrt(252) * 100
    var95 = -np.percentile(pr, 5) * 100
    sharpe = (pr.mean() * 252 - 0.03) / (pr.std() * np.sqrt(252))
    print(f"{name}:")
    print(f"  Annual return:  {ann_r:.2f}%")
    print(f"  Annual vol:     {ann_v:.2f}%")
    print(f"  Sharpe ratio:   {sharpe:.3f}")
    print(f"  95% VaR (1-day): {var95:.3f}%")

portfolio_summary(weights_eq, "Equal-Weight")
print()
portfolio_summary(res_sharpe.x, "Max-Sharpe")
print("\nMax-Sharpe has a better return/risk trade-off (higher Sharpe) but not necessarily lower absolute VaR,")
print("because it concentrates in higher-return (and potentially higher-vol) assets.")

---
## Summary

| Task | Tool |
|---|---|
| Download prices | `yf.download(tickers, start, end)` |
| Log returns | `np.log(prices / prices.shift(1))` |
| Rolling vol | `.rolling(window).std() * √252` |
| Efficient frontier | `scipy.optimize.minimize` with equality constraints |
| VaR | `np.percentile(returns, 1 - alpha)` |
| Expected shortfall | `returns[returns <= VaR].mean()` |

**Next:** NB_11_Econophysics.ipynb — power laws, fat tails, and statistical physics of economic systems.